# The "Wall of Silence": Train-Test Splitting 🧱

In Machine Learning, we aren't just trying to "fit" data; we are trying to **Generalize**. If a model only works on data it has already seen, it is useless for making future predictions. 

**Fitting** is like memorizing the answers to a specific quiz. **Generalization** is like learning the underlying logic so you can answer *any* quiz on that subject. Train-test splitting is how we prove our model can generalize.

---

## 🧠 The Deep Dive: The Concept

### 1. The Ratio (The 80/20 Rule)
*   **Training Set (80%)**: Large enough so the model has enough examples to learn the complex patterns (like the relationship between `Fare` and `Subscription`).
*   **Test Set (20%)**: Small enough to be efficient, but large enough to provide a statistically significant "grade." If your test set is too small (e.g., only 5 rows), one lucky guess could make your model look much better than it is.

### 2. Features (X) vs. Target (y)
Before splitting, we must slice our data vertically to separate the "Hints" from the "Result."

*   **`X` (The Features/Matrix)**: Think of these as the **Clues**. We use a capital `X` because this represents a group of multiple columns (e.g., *Hours Studied, Attendance, Previous Grades*).
*   **`y` (The Target/Vector)**: This is the **Secret Answer** you want to guess. We use a lowercase `y` because it is always just one single column (e.g., *Passed Yes/No*).

> **Why separate?** If we didn't, the model would simply "see" the answer inside the clues and wouldn't have to learn any real logic—it would just memorize that "If the Result says Yes, then the answer is Yes."

### 3. The `random_state` Parameter (The "Save Game" Button)
This is the "Seed" for the random number generator. Since the computer shuffles the rows before splitting, you want that shuffle to be consistent.

*   **The Problem**: Without a seed, every time you run your code, you get a *different* random 80%. If your accuracy changes from 85% to 82%, you won't know if your code actually got worse or if the newest random shuffle was just "unlucky."
*   **The Solution**: Setting `random_state=42` locks the shuffling pattern. Now, every single time you hit "Play," you get the **exact same rows** in your Training set. 
*   **Why 42?**: It's a data science tradition (a reference to *The Hitchhiker's Guide to the Galaxy*), but you can use any number you like! The goal is consistency, not the specific number.

---

## 🏗️ The "Exam" Analogy
*   **Training Set (X_train, y_train)**: This is the "Textbook." The model studies this data to find patterns and rules.
*   **Test Set (X_test, y_test)**: This is the "Final Exam." We hide this data from the model while it's studying. We only show it to the model at the very end to see if it actually learned or if it just memorized the textbook.

---

## 🛠️ The Implementation (Scikit-Learn)

In your notebook, we use the `train_test_split` function. It splits your data into **4 distinct variables**.

```python
from sklearn.model_selection import train_test_split

# 1. Separate Features (X) from Target (y)
# X = everything except the column we want to predict
# y = the column we want to predict (e.g., 'Target' or 'Price')
X = dataset_final.drop('Subscription_Encoded', axis=1) # Example
y = dataset_final['Subscription_Encoded']

# 2. Perform the Split
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2,      # 20% for testing, 80% for training
    random_state=42     # Ensures the split is the same every time you run it
)

# 3. Verify
print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
```

---

## 🧱 The 4 Variables Explained

| Variable | Type | Purpose |
| :--- | :--- | :--- |
| **`X_train`** | Features | The "Questions" the model uses for studying. |
| **`y_train`** | Target | The "Answers" the model uses to correct itself while studying. |
| **`X_test`** | Features | The "Final Exam Questions" (Hidden from model). |
| **`y_test`** | Target | The "Answer Key" used to grade the model's performance. |

---

## ⚠️ Critical Rule: The "Golden Layer"
Once you have split your data, you **never** look at `X_test` or `y_test` during your analysis. Any statistics (like calculating the mean or finding features) must be done on `X_train` only.

> [!IMPORTANT]
> **random_state=42**: This is a convention! You can use any number, but using 42 ensures that if you share your code, someone else will get the exact same rows in their training/test sets as you.


### Test

In [1]:
print("Hello World")

Hello World


## All of the pre-processing

In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, MinMaxScaler

# 1. Load the Fresh Data
df = pd.read_csv('../../data/messy_ml_data.csv')

# ---------------------------------------------------------
# 2. DROP DUPLICATES (THE SEMANTIC WAY)
# ---------------------------------------------------------
# We check all columns except PassengerId to find identical people
search_cols = df.columns.difference(['PassengerId'])
df = df.drop_duplicates(subset=search_cols, keep='first').reset_index(drop=True)

# 3. Drop Metadata
# (PassengerId is gone now, along with the others)
df = df.drop(['PassengerId', 'Email', 'Phone', 'Remarks', 'JoinDate'], axis=1)

# 4. Standardize Text & Mapping
for col in ['Gender', 'City', 'Subscription', 'IsActive']:
    df[col] = df[col].astype(str).str.strip()

active_map = {'True': 1, '1': 1, '1.0': 1, 'Yes': 1, 'False': 0, '0': 0, '0.0': 0, 'No': 0, 'none': 0, 'nan': 0}
df['IsActive'] = df['IsActive'].map(active_map).fillna(0).astype(int)

gender_map = {
    'F': 'Female',
    'female': 'Female',
    'M': 'Male',
    'male': 'Male'
}
df['Gender'] = df['Gender'].map(gender_map).fillna(df['Gender'])

sub_map = {'FREE': 'Free', 'Free': 'Free', 'NONE': 'Free', 'none': 'Free'
           , 'BASIC': 'Basic', 'Basic': 'Basic', 'PREMIUM': 'Premium', 'Premium': 'Premium'}
df['Subscription'] = df['Subscription'].map(sub_map).fillna('Free')

# 5. Handle All Remaining Nulls
df['Score'] = pd.to_numeric(df['Score'], errors='coerce')

for col in ['Age', 'Score', 'Fare']:
    df[col] = df[col].fillna(df[col].median())

df['City'] = df['City'].replace('nan', df['City'].mode()[0])

# ---------------------------------------------------------
# 6. Handle Outliers (Important for clean splitting)
# ---------------------------------------------------------
def remove_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    return data[(data[column] >= (Q1 - 1.5 * IQR)) & (data[column] <= (Q3 + 1.5 * IQR))]

df = remove_outliers(df, 'Age')
df = remove_outliers(df, 'Score')
df = df.reset_index(drop=True)

# 7. Encoding & Scaling
# (We fix the boolean types AND the Fare scaling here)

# Encoding categorical columns
df = pd.get_dummies(df, columns=['Gender', 'City', 'DeptCode', 'DiscountCode'], drop_first=True)

# Ordinal Encoding for your Target
encoder = OrdinalEncoder(categories=[['Free', 'Basic', 'Premium']])
df['Subscription_Encoded'] = encoder.fit_transform(df[['Subscription']])

# Create the final dataframe and reset index
df_final = df.copy().reset_index(drop=True)

# A. Handle the Fare Scaling (Log + Standardize)
df_final['Fare'] = np.log1p(df_final['Fare'])
df_final[['Fare']] = StandardScaler().fit_transform(df_final[['Fare']])

# B. Scale Age and Score
df_final[['Age']] = MinMaxScaler().fit_transform(df_final[['Age']])
df_final[['Score']] = StandardScaler().fit_transform(df_final[['Score']])

# C. Convert all Boolean columns to 1 and 0 (Universal compatibility)
df_final = df_final.apply(lambda x: x.astype(int) if x.dtype == 'bool' else x)

print("🚀 Pipeline Complete: Everything is Numeric and Scaled!")
df_final.head()



🚀 Pipeline Complete: Everything is Numeric and Scaled!


,Age,Fare,Subscription,Score,IsActive,Gender_Male,City_London,City_New York,City_Paris,City_Tokyo,DeptCode_D02,DeptCode_D03,DeptCode_D04,DiscountCode_SAVE20,DiscountCode_WELCOME50,Subscription_Encoded
0,0.614458,-0.808391,Free,-1.009482,1,0,0,0,0,1,0,0,1,0,0,0.0
1,0.168675,-2.343967,Premium,0.892396,0,0,1,0,0,0,0,0,1,0,0,2.0
2,0.855422,-0.953136,Basic,1.455915,0,0,0,1,0,0,1,0,0,0,0,1.0
3,0.722892,-0.383542,Basic,1.244595,1,1,1,0,0,0,0,0,1,0,1,1.0
4,0.240964,-0.901125,Basic,0.258436,0,1,0,0,0,1,0,0,0,0,0,1.0


## After pre-processing analysis and inspection

In [26]:
df_final.isnull().sum()

Age                       0
Fare                      0
Subscription              0
Score                     0
IsActive                  0
Gender_Male               0
City_London               0
City_New York             0
City_Paris                0
City_Tokyo                0
DeptCode_D02              0
DeptCode_D03              0
DeptCode_D04              0
DiscountCode_SAVE20       0
DiscountCode_WELCOME50    0
Subscription_Encoded      0
dtype: int64

In [27]:
df_final.shape

(749, 16)

In [28]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 749 entries, 0 to 748
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Age                     749 non-null    float64
 1   Fare                    749 non-null    float64
 2   Subscription            749 non-null    str    
 3   Score                   749 non-null    float64
 4   IsActive                749 non-null    int64  
 5   Gender_Male             749 non-null    int64  
 6   City_London             749 non-null    int64  
 7   City_New York           749 non-null    int64  
 8   City_Paris              749 non-null    int64  
 9   City_Tokyo              749 non-null    int64  
 10  DeptCode_D02            749 non-null    int64  
 11  DeptCode_D03            749 non-null    int64  
 12  DeptCode_D04            749 non-null    int64  
 13  DiscountCode_SAVE20     749 non-null    int64  
 14  DiscountCode_WELCOME50  749 non-null    int64  
 15  

In [40]:
df_final[['Fare','Age','Score']].describe().round(2)

,Fare,Age,Score
count,749.00,749.00,749.00
mean,0.00,0.48,-0.00
std,1.00,0.28,1.00
min,-3.41,0.00,-2.84
25%,-0.56,0.24,-0.73
50%,0.30,0.48,0.05
75%,0.78,0.72,0.75
max,1.15,1.00,2.86
